In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By

import requests
from bs4 import BeautifulSoup

import pandas as pd

In [16]:
def get_qa_data(soup): 
    qa_list = soup.find_all('div', class_ = 'qi-item')

    qa_data = []
    for qa in qa_list:
        qa_header = qa.find('div', class_ = 'qi-header-top')
        name = qa_header.find('span', class_ = 'qi-name').text
        email = qa_header.find('span', class_ = 'qi-email').text
        id = qa_header.find('span', class_ = 'qi-id-badge').text

        question = qa.find('div', 'qi-question').text

        meta_item = qa.find('div', class_ = 'qi-meta')
        meta = [i.text.strip() for i in meta_item.find_all('span', class_ = 'qi-meta-item')]

        qa_body = qa.find('div', class_ = 'qi-body')
        answer = qa_body.find('div', class_ = 'qi-reply').text.strip()
        rates = [i.text for i in qa_body.find('div', class_ = 'qi-rate-bar').find_all('span')[:2]]

        data = {
            'name' : name, 
            'email' : email, 
            'id' : id, 
            'meta' : meta, 
            'question' : question, 
            'answer' : answer, 
            'rates' : rates
        }

        qa_data.append(data)

    print(f'Scraped {len(qa_data)} questions')
    return qa_data

In [22]:
backup = scraped_data[::]
len(backup)/20

286.0

In [26]:
# url = 'https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=1'

page = 291
# scraped_data = []

while page < 300:
    url =  f'https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page={page}'
    response = requests.get(url)

    if(response.status_code == 500):
        print(f'Couldnt scrape page but continueing: {url}, status code: {response.status_code}') 
        page += 1
        continue
    elif(response.status_code != 200):
        print(f'Couldnt scrape page: {url}, status code: {response.status_code}') 
        break
    else: 
        print(f'Successfully fetched page: {url}') 

    soup = BeautifulSoup(response.text, 'html.parser')

    qa_data = get_qa_data(soup)

    if(len(qa_data) == 0):
        print('There is no data on this page!')
        break

    scraped_data.extend(qa_data)

    page += 1


Couldnt scrape page but continueing: https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=291, status code: 500
Successfully fetched page: https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=292
Scraped 20 questions
Couldnt scrape page but continueing: https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=293, status code: 500
Successfully fetched page: https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=294
Scraped 12 questions
Successfully fetched page: https://www.taxes.gov.az/az/page/suallar-ve-cavablar?page=295
Scraped 0 questions
There is no data on this page!


In [33]:
df.to_csv("../output/scraped/raw_data.csv", index=False, encoding="utf-8-sig")

# Analysis Part

In [5]:
soup = BeautifulSoup(response.text, 'html.parser')

In [6]:
qa_list = soup.find_all('div', class_ = 'qi-item')
len(qa_list)

20

In [7]:
qa = qa_list[0]
qa_header = qa.find('div', class_ = 'qi-header-top')
name = qa_header.find('span', class_ = 'qi-name').text
email = qa_header.find('span', class_ = 'qi-email').text
id = qa_header.find('span', class_ = 'qi-id-badge').text

question = qa.find('div', 'qi-question').text

meta_item = qa.find('div', class_ = 'qi-meta')
meta = [i.text.strip() for i in meta_item.find_all('span', class_ = 'qi-meta-item')]

qa_body = qa.find('div', class_ = 'qi-body')
answer = qa_body.find('div', class_ = 'qi-reply').text.strip()
rates = [i.text for i in qa_body.find('div', class_ = 'qi-rate-bar').find_all('span')[:2]]

data = {
    'name' : name, 
    'email' : email, 
    'id' : id, 
    'meta' : meta, 
    'question' : question, 
    'answer' : answer, 
    'rates' : rates
}

In [8]:
qa_data = []
for qa in qa_list:
    qa_header = qa.find('div', class_ = 'qi-header-top')
    name = qa_header.find('span', class_ = 'qi-name').text
    email = qa_header.find('span', class_ = 'qi-email').text
    id = qa_header.find('span', class_ = 'qi-id-badge').text

    question = qa.find('div', 'qi-question').text

    meta_item = qa.find('div', class_ = 'qi-meta')
    meta = [i.text.strip() for i in meta_item.find_all('span', class_ = 'qi-meta-item')]

    qa_body = qa.find('div', class_ = 'qi-body')
    answer = qa_body.find('div', class_ = 'qi-reply').text.strip()
    rates = [i.text for i in qa_body.find('div', class_ = 'qi-rate-bar').find_all('span')[:2]]

    data = {
        'name' : name, 
        'email' : email, 
        'id' : id, 
        'meta' : meta, 
        'question' : question, 
        'answer' : answer, 
        'rates' : rates
    }

    qa_data.append(data)

In [10]:
df = pd.DataFrame(qa_data)